# 06 · Avaliação — qual modelo escolher e como ele se sai num mês novo
Lê os cinco algoritmos treinados no 05_train direto do MLflow, compara com uma referência simples, escolhe o modelo e avalia no out-of-time: curva ROC, calibração das probabilidades, precisão no top 50 e importância das features por bloco.

**Entrada:** experimento `churn-teo-me-why` (MLflow) e `abt_churn_bases_v0` \
**Saída:** figuras 12_* a 14_* e, em output/tables/, comparacao_modelos_v0.csv, estabilidade_modelos_v0.csv, metricas_modelo_final_v0.csv e top50_churn_v0.csv

In [0]:
%pip install feature_engine xgboost lightgbm -q

In [0]:
%run ./00_setup

## 0. Leitura das bases
Carrega os modelos do MLflow e as bases de treino, teste e out-of-time salvas pelo 05_train. Nada é sorteado ou treinado de novo.

In [0]:
import os
import mlflow
import logging
os.environ['MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR'] = 'false'  # esconde as barras de download dos modelos
os.environ['MLFLOW_DATABRICKS_TAGS_TO_SKIP'] = 'extraContext'  
logging.getLogger('mlflow.tracking.context.registry').setLevel(logging.ERROR)

import mlflow
from sklearn import base, metrics, pipeline

mlflow.sklearn.autolog(disable=True)

In [0]:


VERSAO      = 'v0'                                   # mesma versão do 05_treino
EXPERIMENTO = f'/Users/{USUARIO}/churn-teo-me-why'    # experimento do MLflow

mlflow.set_experiment(EXPERIMENTO)

# bases salvas pelo 05_treino (seção 1.5): nada é sorteado ou treinado de novo
bases = spark.table(f'{TABELA_ABT}_bases_{VERSAO}').toPandas()
features = bases.columns.drop(['IdCliente', 'dtRef', 'churn', 'base']).tolist()

treino = bases[bases['base'] == 'treino']
teste  = bases[bases['base'] == 'teste']
df_oot = bases[bases['base'] == 'oot']

X_train, y_train = treino[features], treino['churn']
X_test,  y_test  = teste[features],  teste['churn']
X_oot,   y_oot   = df_oot[features], df_oot['churn']

## 1. Referência de comparação

In [0]:
acc_todos_saem = metrics.accuracy_score(y_test, [1] * len(y_test))   # churn é a classe majoritária (73%)
auc_recencia   = metrics.roc_auc_score(y_test, X_test['recencia'])
print(f'Chutar "todo mundo sai": acurácia {acc_todos_saem:.3f}')
print(f'Recência sozinha: AUC {auc_recencia:.3f}')

## 2. Comparação dos algoritmos
Métricas dos cinco runs, lidas do experimento do MLflow. A escolha usa a AUC da validação cruzada: com cinco modelos, escolher pelo teste favorece quem acertou por acaso.

In [0]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENTO],
    filter_string=f"attributes.status = 'FINISHED' and tags.dt_oot = '{DT_OOT}'",
    order_by=['start_time DESC'],
)
runs = runs[~runs['tags.mlflow.runName'].str.startswith('final_')]          # só runs com as métricas
if 'tags.versao' in runs.columns and (runs['tags.versao'] == VERSAO).any():  # só a versão atual, se marcada
    runs = runs[runs['tags.versao'] == VERSAO]
runs = runs.drop_duplicates(subset='tags.mlflow.runName', keep='first')     # o mais recente de cada algoritmo

comparacao = (
    runs.rename(columns={
        'tags.mlflow.runName': 'modelo',
        'metrics.auc_train':   'auc_treino',
        'metrics.auc_cv':      'auc_cv',
        'metrics.auc_test':    'auc_teste',
    })
    .set_index('modelo')[['auc_treino', 'auc_cv', 'auc_teste', 'run_id']]
)
comparacao['diferenca'] = comparacao['auc_treino'] - comparacao['auc_teste']
comparacao['ganho_vs_chute'] = comparacao['auc_teste'] / 0.5 - 1
comparacao.loc['referência: recência', 'auc_teste'] = auc_recencia
comparacao = comparacao.sort_values('auc_cv', ascending=False)

comparacao.drop(columns='run_id').to_csv(f'{TABLE_DIR}/comparacao_modelos_{VERSAO}.csv')
comparacao.round(3)

## 3. Escolha do modelo

O Random Forest teve a maior AUC na validação cruzada (0,819), à frente da regressão logística (0,805), e também a maior no out-of-time (0,868, seção 4.2). Traz ainda a importância das features pronta

In [0]:
MELHOR = comparacao.drop("referência: recência")["auc_cv"].idxmax()


modelo_final = mlflow.sklearn.load_model(f"runs:/{comparacao.loc[MELHOR, 'run_id']}/model")
print(f"Modelo escolhido: {MELHOR}")

### 3.1 Curva ROC dos 4 melhores no teste
Cada curva mostra, para cada ponto de corte, quanto o modelo acerta de quem sai (eixo vertical) contra quantos alarmes falsos ele dá (eixo horizontal). Quanto mais perto do canto superior esquerdo, melhor.

In [0]:
top4 = comparacao.drop('referência: recência').head(4).index

fig, ax = plt.subplots(figsize=(7, 6))
for nome_modelo, cor in zip(top4, BLUE_PALETTE):
    modelo_run = mlflow.sklearn.load_model(f"runs:/{comparacao.loc[nome_modelo, 'run_id']}/model")
    prob = modelo_run.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = metrics.roc_curve(y_test, prob)
    ax.plot(fpr, tpr, color=cor, lw=2, label=f'{nome_modelo} (AUC {metrics.roc_auc_score(y_test, prob):.3f})')

fpr, tpr, _ = metrics.roc_curve(y_test, X_test['recencia'])
ax.plot(fpr, tpr, color=COR_NEUTRA, lw=1.5, ls='--', label=f'recência sozinha (AUC {auc_recencia:.3f})')
ax.plot([0, 1], [0, 1], color=COR_NEUTRA, lw=1, ls=':', label='chute (AUC 0,5)')

ax.set(title='Curva ROC no teste: os melhores modelos praticamente empatam',
       xlabel='taxa de falsos positivos (quem ficou, mas o modelo disse que sai)',
       ylabel='taxa de verdadeiros positivos (quem saiu e o modelo acertou)')
ax.legend(loc='lower right')
salvar(fig, '12_roc_comparacao_modelos')

## 4. Assess: desempenho no out-of-time
O teste definitivo: um mês (jun/26) que o modelo nunca viu, com todos os usuários ativos, e não uma safra sorteada por usuário.

### 4.1 Curva ROC em treino, teste e out-of-time
Se as curvas ficarem próximas, o modelo não decorou o treino e se mantém no tempo.

In [0]:
fig, ax = plt.subplots(figsize=(7, 6))
for nome_base, X_base, y_base, cor in [
    ('treino',      X_train, y_train, BLUE_PALETTE[0]),
    ('teste',       X_test,  y_test,  BLUE_PALETTE[1]),
    ('out of time', X_oot,   y_oot,   BLUE_PALETTE[2]),
]:
    prob = modelo_final.predict_proba(X_base)[:, 1]
    fpr, tpr, _ = metrics.roc_curve(y_base, prob)
    ax.plot(fpr, tpr, color=cor, lw=2, label=f'{nome_base} (AUC {metrics.roc_auc_score(y_base, prob):.3f})')

ax.plot([0, 1], [0, 1], color=COR_NEUTRA, lw=1, ls=':', label='chute (AUC 0,5)')
ax.set(title=f'{NOME_MODELO.get(MELHOR, MELHOR)}: a AUC se mantém no mês novo',
       xlabel='taxa de falsos positivos', ylabel='taxa de verdadeiros positivos')
ax.legend(loc='lower right')
salvar(fig, '13_roc_modelo_final')

#### Acurácia e AUC com ganho sobre o chute 
Ganho de acurácia: contra chutar a classe mais comum. Ganho de AUC: contra 0,5 (sorteio)

In [0]:
linhas = []
for nome_base, X_base, y_base in [('treino', X_train, y_train), ('teste', X_test, y_test), ('oot', X_oot, y_oot)]:
    pred = modelo_final.predict(X_base)
    prob = modelo_final.predict_proba(X_base)[:, 1]
    acc_chute = metrics.accuracy_score(y_base, [int(y_base.mean() >= 0.5)] * len(y_base))
    acc = metrics.accuracy_score(y_base, pred)
    auc = metrics.roc_auc_score(y_base, prob)
    linhas.append({'base': nome_base, 'acuracia': acc, 'ganho_acuracia': acc / acc_chute - 1,
                   'auc': auc, 'ganho_auc': auc / 0.5 - 1})
metricas_final = pd.DataFrame(linhas).set_index('base')
metricas_final.to_csv(f'{TABLE_DIR}/metricas_modelo_final_{VERSAO}.csv')
metricas_final.round(3)

### 4.2 Os 5 modelos no out-of-time
A AUC sobe no out-of-time para todos os modelos porque a base é diferente: todos os ativos do mês, e não uma safra sorteada por usuário. O que importa é a ordem, e o Random Forest se mantém no topo.

In [0]:
# ---------- AUC dos 5 modelos no teste e no out-of-time ----------
# só para conferir a estabilidade no tempo: a escolha continua sendo pela AUC cv (seção 3)
estabilidade = []
for nome_modelo in comparacao.drop('referência: recência').index:
    modelo_run = mlflow.sklearn.load_model(f"runs:/{comparacao.loc[nome_modelo, 'run_id']}/model")
    auc_teste = metrics.roc_auc_score(y_test, modelo_run.predict_proba(X_test)[:, 1])
    auc_oot   = metrics.roc_auc_score(y_oot,  modelo_run.predict_proba(X_oot)[:, 1])
    estabilidade.append({'modelo': nome_modelo,
                         'auc_cv': comparacao.loc[nome_modelo, 'auc_cv'],
                         'auc_teste': auc_teste,
                         'auc_oot': auc_oot,
                         'diferenca_oot_teste': auc_oot - auc_teste})

estabilidade = pd.DataFrame(estabilidade).set_index('modelo').sort_values('auc_cv', ascending=False)
estabilidade.to_csv(f'{TABLE_DIR}/estabilidade_modelos_{VERSAO}.csv')
estabilidade.round(3)

### 4.3 Calibração das probabilidades
A probabilidade prevista bate com o churn real? O treino tem 73% de churn (uma safra por usuário) e o out-of-time, 44%. Se o modelo herdou a taxa do treino, as probabilidades saem altas demais.

In [0]:
prob_oot = modelo_final.predict_proba(X_oot)[:, 1]
print(f'Probabilidade média prevista: {prob_oot.mean():.1%} | churn real no out-of-time: {y_oot.mean():.1%}')

# por faixa de probabilidade: prevista × real
calib = (pd.DataFrame({'prob': prob_oot, 'real': y_oot.values})
           .assign(faixa=lambda d: pd.qcut(d['prob'], 5, duplicates='drop'))
           .groupby('faixa', observed=True)
           .agg(usuarios=('real', 'size'), prob_media=('prob', 'mean'), churn_real=('real', 'mean')))
calib.round(3)

### 4.4 Precisão no top 50
Dos 50 usuários com maior probabilidade de churn no out-of-time, quantos realmente saíram. É a métrica que responde à pergunta de negócio.


In [0]:
ranking_oot = (df_oot[['IdCliente', 'churn']]
               .assign(prob_churn=prob_oot)
               .sort_values('prob_churn', ascending=False))

precisao_top50 = ranking_oot.head(50)['churn'].mean()
print(f'Precisão no top 50: {precisao_top50:.0%} | churn médio do out-of-time: {y_oot.mean():.0%} '
      f'| ganho: {precisao_top50 / y_oot.mean():.1f}x')

## 5. Importância das features por bloco
Quais blocos do MECE o modelo mais usa para prever o churn.

In [0]:
BLOCOS = {   # C vem primeiro: diasAtivosD28Anterior também contém 'diasAtivos'
    'C. Tendência':  ['diasAtivosD28Anterior', 'deltaDiasD28', 'tendencia_'],
    'A. Perfil':     ['diasDesdePrimeira', 'flNovatoRisco', 'qtdRedes'],
    'B. Volume':     ['recencia', 'diasAtivos', 'qtdTransacoesD28', 'pontosGanhosD28', 'freqVida',
                      'transacoesPorDiaD28', 'horasPorDiaD28', 'percentilPontos', 'zScorePontos'],
    'D. Mix':        ['share_', 'produtoDominante', 'qtdProdutosDistintos'],
    'E. Hábito':     ['qtdMadrugada', 'qtdManha', 'qtdTarde', 'qtdNoite', 'qtdDiasSemana', 'periodoDominante',
                      'Intervalo', 'qtdRetornos', 'razaoAtraso', 'fezStreak', 'diasUltimoStreak'],
    'F. Comunidade': ['audienciaMedia', 'pctLivesPresente', 'livesPerdidas'],
    'G. Loja':       ['pontosGastosD28', 'saldoPontos', 'gastouD28'],
    'H. Educação':   ['Curso', 'pctMedioCompleto', 'qtdEpsD28', 'diasDesdeUltimoEp'],
}
def bloco(col):
    for b_, padroes in BLOCOS.items():
        if any(p in col for p in padroes):
            return b_
    return 'outros'

clf_final = modelo_final[-1]
if isinstance(clf_final, pipeline.Pipeline):   # modelos com escala: o classificador é o último passo
    clf_final = clf_final[-1]
colunas = modelo_final[:-1].transform(X_train.head()).columns

if hasattr(clf_final, 'feature_importances_'):
    peso = clf_final.feature_importances_
else:
    peso = np.abs(clf_final.coef_[0])

importancia = pd.DataFrame({'feature': colunas, 'importancia': peso / peso.sum()})
importancia['bloco'] = importancia['feature'].map(bloco)
por_bloco = importancia.groupby('bloco')['importancia'].sum().sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(por_bloco.index, por_bloco * 100, color=COLOR_BLUE)
ax.bar_label(ax.containers[0], labels=[f'{v:.0%}' for v in por_bloco], padding=3, color=COLOR_TITLE)
ax.set(title=f'Importância das features por bloco ({MELHOR})', xlabel='% da importância total', ylabel='')
salvar(fig, '14_importancia_por_bloco')

importancia.sort_values('importancia', ascending=False).head(15).round(3)

### 5.1 Quem sai × quem fica nas features mais importantes 
Média de cada feature entre quem saiu e quem ficou. Razão abaixo de 1: quem sai tem menos; acima de 1: tem mais.

In [0]:
X_train_transform = modelo_final[:-1].transform(X_train)
num_cols = X_train_transform.select_dtypes('number').columns
df_train = X_train_transform[num_cols].copy()
df_train['churn'] = y_train.values

sumario = df_train.groupby('churn')[num_cols].mean().T
sumario.columns = ['media_fica', 'media_sai']
sumario['razao_sai_fica'] = sumario['media_sai'] / sumario['media_fica']
sumario['importancia'] = sumario.index.map(importancia.set_index('feature')['importancia'])
sumario = sumario.sort_values('importancia', ascending=False)
sumario.to_csv(f'{TABLE_DIR}/perfil_churn_{VERSAO}.csv')
sumario.head(15).round(3)

## Correlação entre as features mais importantes

In [0]:
abt = carregar_abt(base='treino')
abt = abt.toPandas() if hasattr(abt, 'toPandas') else abt

top5 = ['pctLivesPresente', 'livesPerdidas', 'recencia', 'diasAtivosD7', 'diasAtivosD14']
abt[top5].corr(method='spearman').round(2)

Quatro das cinco variáveis mais importantes são muito correlacionadas (recência e lives perdidas: 0,96; dias ativos: 0,84 a 0,87). Isso não prejudica a previsão do Random Forest, mas divide a importância entre elas. Por isso leio por tema: o churn é explicado por dois sinais, o afastamento recente e o hábito de ir às lives. Tratar a redundância fica para a v1.

## 6. Modelo final
Retreina o modelo escolhido com treino + teste e salva no MLflow, junto com as métricas do out-of-time. As métricas são do modelo treinado só no treino (o que foi avaliado); o modelo salvo é o retreinado, que vai para produção.

In [0]:
from sklearn import base
from mlflow.models import infer_signature

mlflow.sklearn.autolog(disable=True)
mlflow.set_experiment(EXPERIMENTO)

X_tudo = pd.concat([X_train, X_test])
y_tudo = pd.concat([y_train, y_test])

# exemplo de entrada com inteiros como float: evita o aviso de schema
exemplo = X_tudo.head().copy()
cols_int = exemplo.select_dtypes('integer').columns
exemplo[cols_int] = exemplo[cols_int].astype('float64')

# métricas do modelo avaliado (treinado só no treino)
metricas_log = {'auc_cv': comparacao.loc[MELHOR, 'auc_cv'], 'precisao_top50': precisao_top50}
for sufixo, X_base, y_base in [('train', X_train, y_train), ('test', X_test, y_test), ('oot', X_oot, y_oot)]:
    metricas_log[f'auc_{sufixo}'] = metrics.roc_auc_score(y_base, modelo_final.predict_proba(X_base)[:, 1])
    metricas_log[f'acc_{sufixo}'] = metrics.accuracy_score(y_base, modelo_final.predict(X_base))

with mlflow.start_run(run_name=f'final_{MELHOR}_{VERSAO}') as run:
    mlflow.set_tags({'versao': VERSAO, 'tipo': 'final', 'modelo_base': MELHOR})
    mlflow.log_metrics(metricas_log)

    modelo_prod = base.clone(modelo_final).fit(X_tudo, y_tudo)

    # assinatura explícita: saída como float, evita o aviso de coluna inteira
    assinatura = infer_signature(exemplo, modelo_prod.predict(exemplo).astype('float64'))
    mlflow.sklearn.log_model(modelo_prod, name='model', input_example=exemplo, signature=assinatura)

# ---------- Registro no Unity Catalog ----------
mlflow.set_registry_uri('databricks-uc')
registro = mlflow.register_model(f'runs:/{run.info.run_id}/model', MODELO_REGISTRADO)
mlflow.MlflowClient().set_registered_model_alias(MODELO_REGISTRADO, 'champion', registro.version)
print(f'Registrado: {MODELO_REGISTRADO} v{registro.version} (alias champion)')

pd.Series(metricas_log).round(4)

## 7. Top 50 dos próximos 28 dias
Aplica o modelo final aos usuários ativos na safra de set/26 (DT_SCORE) e lista os 50 com maior probabilidade de churn nos 28 dias a partir de 01/09. Salvo em output/tables/top50_churn_v0.csv

In [0]:
abt = spark.table(TABELA_ABT).toPandas()
dt_score = abt['dtRef'].max()
base_score = abt[abt['dtRef'] == dt_score].copy()
base_score['prob_churn'] = modelo_prod.predict_proba(base_score[features])[:, 1]

top50 = (base_score[['IdCliente', 'dtRef', 'prob_churn']]
         .sort_values('prob_churn', ascending=False).head(50).reset_index(drop=True))
top50.index += 1
top50.to_csv(f'{TABLE_DIR}/top50_churn_{VERSAO}.csv')
top50